# Deploy Apex metric views

Job task that creates the three YAML 1.1 metric views in the target schema.


## Bind target catalog and schema


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Create views from bundled YAML


In [ ]:
from pathlib import Path

mv_dir = root / "level-2-performance-lab" / "metric_views"
if not mv_dir.exists():
    mv_dir = root / "src" / "metric_views"
for file_name, view_name in [
    ("mv_line_performance.yaml", "mv_line_performance"),
    ("mv_quality.yaml", "mv_quality"),
    ("mv_maintenance.yaml", "mv_maintenance"),
]:
    yaml_text = (mv_dir / file_name).read_text().replace("${catalog}", cfg.catalog).replace("${schema}", cfg.schema)
    spark.sql(
        f"""CREATE OR REPLACE VIEW {cfg.table(view_name)}
WITH METRICS
LANGUAGE YAML
AS $$
{yaml_text}
$$"""
    )
    print("created", cfg.table(view_name))
print("PASS metric views")
